X_test = [
  [1.5, 1.5],
  [5.5, 5.5]
]
X_train = [
  [1, 1],
  [2, 2],
  [3, 3],
  [4, 4],
  [5, 5],
  [6, 6]
]
y_train = [
  4,
  8,
  12,
  16,
  20,
  24
]

In [ ]:
import numpy as np

class DecisionTreeRegressor:
    def __init__(self, max_depth=3, min_samples_split=4):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    class _Node:
        def __init__(self, value=None, feature=None, thr=None,
                     left=None, right=None):
            self.value = value
            self.feature = feature
            self.thr = thr
            self.left = left
            self.right = right

    def fit(self, X, y):
        self.root = self._build(X, y, 0)

    def predict(self, X):
        return [self._predict_row(row, self.root) for row in X]

    def _predict_row(self, row, node):
        while node.feature is not None:
            if row[node.feature] <= node.thr:
                node = node.left
            else:
                node = node.right
        return node.value

    def _build(self, X, y, depth):
        n = len(y)
        mean_y = sum(y) / n
        if (depth >= self.max_depth or n < self.min_samples_split or
                self._variance(y) == 0.0):
            return self._Node(value=mean_y)

        best = self._best_split(X, y)
        if best is None:
            return self._Node(value=mean_y)

        feat, thr, left_idx, right_idx = best
        left_X = [X[i] for i in left_idx]
        left_y = [y[i] for i in left_idx]
        right_X = [X[i] for i in right_idx]
        right_y = [y[i] for i in right_idx]
        left_node = self._build(left_X, left_y, depth + 1)
        right_node = self._build(right_X, right_y, depth + 1)
        return self._Node(feature=feat, thr=thr,
                          left=left_node, right=right_node)

    def _best_split(self, X, y):
        n = len(y)
        n_feat = len(X[0])
        best_mse = None
        best_feat = None
        best_thr = None
        best_left = None
        best_right = None

        for f in range(n_feat):
            vals = sorted(set(row[f] for row in X))
            if len(vals) <= 1:
                continue
            thrs = [(vals[i] + vals[i + 1]) / 2.0
                    for i in range(len(vals) - 1)]

            for thr in thrs:
                left_idx = [i for i, row in enumerate(X)
                            if row[f] <= thr]
                right_idx = [i for i, row in enumerate(X)
                             if row[f] > thr]
                if not left_idx or not right_idx:
                    continue

                left_y = [y[i] for i in left_idx]
                right_y = [y[i] for i in right_idx]
                mse = (len(left_y) * self._variance(left_y) +
                       len(right_y) * self._variance(right_y)) / n

                if (best_mse is None or mse < best_mse or
                        (mse == best_mse and f < best_feat) or
                        (mse == best_mse and f == best_feat and
                         thr < best_thr)):
                    best_mse = mse
                    best_feat = f
                    best_thr = thr
                    best_left = left_idx
                    best_right = right_idx

        if best_mse is None:
            return None
        return best_feat, best_thr, best_left, best_right

    def _variance(self, y):
        mean_y = sum(y) / len(y)
        return sum((v - mean_y) ** 2 for v in y) / len(y)

# Instantiate model
model = DecisionTreeRegressor()
# Fit model
model.fit(X_train, y_train)
# Get predictions
y_test = model.predict(X_test)
# Output predictions
y_test